# PEMS — Notebook 2: Menus and Actions

**Notebook 1** used menus as data to decide *where to go*. Selecting a client opened Client Workspace, but no client was ever selected.

**This notebook** adds *what to do*. Each menu option gets an **action**: the name of a function that runs when the option is chosen. You will:

1. Rebuild the menus from Notebook 1 into the menus PEMS uses this week, and save them as `menus.json`.
2. Add an `action` to every option.
3. See how `actions.py`, `load_data.py`, and `pems.py` turn an action name into a function call.
4. Learn what `state` holds, then write `create_client` for homework.

**How to work:** Run the cells in order and predict before you run. Nothing in this notebook asks for input, so **Kernel → Restart & Run All** is always safe.

---
## 1. Rebuild the menus

These are the menus you navigated at the end of Notebook 1 (`navigation_only.py`):

In [ ]:
nav_menus = {
    "client_management": {
        "title": "Client Management Menu",
        "options": [
            {"menu_item": "Select Client", "next_menu": "client_workspace"},
            {"menu_item": "Create Client", "next_menu": None},
        ],
    },
    "client_workspace": {
        "title": "Client Workspace",
        "options": [
            {"menu_item": "Transactions", "next_menu": "transactions"},
            {"menu_item": "Reports", "next_menu": "reports"},
        ],
    },
    "transactions": {
        "title": "Transactions",
        "options": [
            {"menu_item": "Contribution", "next_menu": None},
            {"menu_item": "Withdrawal", "next_menu": None},
        ],
    },
    "reports": {
        "title": "Reports",
        "options": [
            {"menu_item": "Holdings", "next_menu": None},
            {"menu_item": "Performance", "next_menu": None},
        ],
    },
}

This week we build only **Select Client** and **Create Client**. Transactions and Reports come in the next two assignments. Until then, each of those menus gets a single placeholder option, so every menu is still reachable and Enter still takes you back:

```
Transactions                 Reports
1) Not Implemented           1) Not Implemented
```

**Your task:** In the next cell, write the new `nav_menus`. Keep `client_management` and `client_workspace` exactly as they are. Replace the options of `transactions` and of `reports` with one option each: `menu_item` = `"Not Implemented"`, `next_menu` = `None`.

In [ ]:
# YOUR CODE — the rebuilt nav_menus (four menus).


In [ ]:
# SUPPLIED VALIDATOR — the Notebook 1 validator, plus one new check:
# with check_actions=True, every option must also have an "action" key
# whose value is None or a function name (a non-empty string).
def validate_menus(menus, check_actions=False):
    valid = True
    if not isinstance(menus, dict):
        print("Menu data must be a dictionary.")
        return False
    if "client_management" not in menus:
        print('Missing starting menu: "client_management"')
        valid = False
    for menu_name, menu in menus.items():
        if not isinstance(menu, dict):
            print(f"{menu_name}: menu definition must be a dictionary")
            valid = False
            continue
        if not isinstance(menu.get("title"), str) or not menu["title"].strip():
            print(f'{menu_name}: missing or invalid "title"')
            valid = False
        options = menu.get("options")
        if not isinstance(options, list) or not options:
            print(f'{menu_name}: "options" must be a non-empty list')
            valid = False
            continue
        for number, option in enumerate(options, 1):
            location = f"{menu_name}, option {number}"
            if not isinstance(option, dict):
                print(f"{location}: option must be a dictionary")
                valid = False
                continue
            label = option.get("menu_item")
            if not isinstance(label, str) or not label.strip():
                print(f'{location}: missing or invalid "menu_item"')
                valid = False
            if "next_menu" not in option:
                print(f'{location}: missing "next_menu"')
                valid = False
            elif option["next_menu"] is not None and option["next_menu"] not in menus:
                print(f"{location}: undefined next_menu: {option['next_menu']}")
                valid = False
            if check_actions:
                if "action" not in option:
                    print(f'{location}: missing "action"')
                    valid = False
                elif option["action"] is not None and (
                        not isinstance(option["action"], str) or not option["action"].strip()):
                    print(f"{location}: action must be a function name or None")
                    valid = False
    if valid:
        print("Menus are valid.")
    return valid


# CHECK — Section 1 task: rebuilt nav_menus.
placeholder = [{"menu_item": "Not Implemented", "next_menu": None}]

if validate_menus(nav_menus):
    for menu_name in ("transactions", "reports"):
        options = nav_menus[menu_name]["options"]
        if options == placeholder:
            print(f"{menu_name}: OK")
        else:
            print(f"{menu_name}: NOT YET. Expected {placeholder}")
            print(f"{' ' * len(menu_name)}  Found    {options}")

---
## 2. Add actions to the menus

Each option now gets a third field, `action`. It follows the same pattern as `next_menu`:

| Field | Value | Meaning |
|---|---|---|
| `next_menu` | a menu **name**, or `None` | where to go after the option is chosen |
| `action` | a function **name**, or `None` | what to do when the option is chosen |

Only two options do work this week:

| Menu | Option | `action` | `next_menu` |
|---|---|---|---|
| `client_management` | Select Client | `"select_client"` | `"client_workspace"` |
| | Create Client | `"create_client"` | `None` |
| `client_workspace` | Transactions | `None` | `"transactions"` |
| | Reports | `None` | `"reports"` |
| `transactions` | Not Implemented | `None` | `None` |
| `reports` | Not Implemented | `None` | `None` |

Select Client has **both**: it runs `select_client`, then opens Client Workspace. Create Client only runs its action and stays on Client Management.

**Your task:** Add an `"action"` key to **every** option in `nav_menus`, using the table. Edit the dictionary in Section 1 (and rerun it), or add the keys with code here.

In [ ]:
# YOUR CODE — add "action" to every option.


In [ ]:
# CHECK — every option has an action; only two are not None.
if validate_menus(nav_menus, check_actions=True):
    for menu_name, menu in nav_menus.items():
        for option in menu["options"]:
            print(f"{menu_name:18} {option['menu_item']:16} action={option['action']!r:18} next_menu={option['next_menu']!r}")

### Save the menus as `menus.json`

PEMS doesn't keep its menus in Python code; `load_data.py` reads them from `data/menus.json`. Instead of typing JSON by hand, let Python write it. `json.dump` converts Python to JSON for you: `None` becomes `null`, and the variable name `nav_menus =` is not part of the file.

In [ ]:
import json

with open("menus.json", "w", encoding="utf-8") as file:
    json.dump(nav_menus, file, indent=4)

with open("menus.json", "r", encoding="utf-8") as file:
    print(file.read()[:500], "...")

**Copy `menus.json` into your project's `data` folder**, replacing the old one. That is the file `load_data.py` reads.

In [ ]:
# QUESTION 1
# A. In menus.json, what is the value of Select Client's "action"? Is it a function?
# B. Why can't menus.json store the select_client function itself?

# YOUR ANSWERS:

---
## 3. Functions are objects

To turn the *name* `"select_client"` into something we can run, we need one new idea: a function is an object, like a list or a dictionary. A name without parentheses **refers** to the function; parentheses **call** it.

In [ ]:
def hello(name):
    print(f"Hello, {name}")

ref = hello              # no parentheses: ref now refers to the same function
print(ref is hello)
print(type(ref))

ref("Bob")               # parentheses: the function runs

Because a function is an object, it can be a **value in a dictionary**. Then a string key leads to a function, the same way a menu name led to a menu in Notebook 1:

In [ ]:
def say_hi(state):
    print("Hi from say_hi")

def say_bye(state):
    print("Bye from say_bye")

greetings = {
    "say_hi": say_hi,       # key: a string    value: the function (no parentheses)
    "say_bye": say_bye,
}

name = "say_bye"            # a name, as it might come from a file
func = greetings[name]      # look up the function by its name
func({})                    # call it

In [ ]:
# QUESTION 2 — Predict, then run to check
# What does each line print, or does it raise an error?
#   print(greetings["say_hi"])
#   greetings["say_hi"]({})
#   greetings["say_hello"]({})

# YOUR PREDICTIONS:

---
## 4. `actions.py`: the functions and the `actions` dictionary

Every action name in `menus.json` needs two things in `actions.py`:

1. **A function**, which always takes one parameter, `state`.
2. **An entry in the `actions` dictionary**: the key is the name *exactly* as it appears in `menus.json`; the value is the function, without parentheses.

```python
actions = {
    "select_client": select_client,
    "create_client": create_client,
}
```

### Worked example: `select_client`

```python
def select_client(state):
    clear()
    state['client_idx'] = None

    clients = state['clients']

    if len(clients) == 0:
        print('You have no clients. Create a client')
        pause(2)
        return False   # returning False will keep you on the same menu
    else:
        for idx, client in enumerate(clients, 1):
            print(f"{idx}) {client['name']}")

        print()
        txt = 'choice: '
        while True:
            entry = input(txt).strip()
            if entry == '':
                return False          # cancel: stay on Client Management
            try:
                choice = int(entry)
                if 1 <= choice <= len(clients):
                    break
            except ValueError:
                pass
            txt = 'Enter a valid choice (Enter to cancel): '

        state['client_idx'] = choice - 1
        
```

What to notice:

- **The return value answers one question: stay on this menu?** `False` means "the action didn't happen; stay here." That happens twice: no clients, and Enter to cancel. On success, Select Client returns `None`, and `pems.py` goes on to open Client Workspace.
- **The real work is a change to `state`.** The selected client is recorded by setting `state['client_idx']`, not by returning it. Section 7 explains `state`.
- **Displayed numbers start at 1; list positions start at 0.** That is why it stores `choice - 1`.

In [ ]:
# QUESTION 3
# A. List the TWO situations in which select_client returns False.
#    What does the user see happen in each?
# B. The user has 3 clients and types 2. What value is stored in
#    state['client_idx']? Why not 2?
# C. Suppose you write select_client but forget to add it to the actions
#    dictionary. Predict what happens (you'll check this in Section 5).

# YOUR ANSWERS:

---
## 5. Binding in `load_data.py`: from name to function

`menus.json` holds the **name** `"select_client"`. `actions.py` has a dictionary from that name to the **function**. Something has to connect them. These five lines in `load_data.py` do it, right after the menus are read:

```python
for menu in menus.values():
    for option in menu["options"]:
        action_name = option["action"]

        if action_name is not None:
            option["action"] = actions[action_name]
```

For every option with an action, the loop looks up the name in `actions` and **replaces the string with the function**. This is called **binding**. It's Notebook 1's indirection again: `menus[name]` found a menu; `actions[name]` finds a function. The difference is timing: binding happens **once**, when PEMS loads, and the function is stored back into the option.

The next cell does exactly this to the `menus.json` you wrote in Section 2. So the cell runs without input, the `actions` dictionary here holds two stand-in functions with the same names.

In [ ]:
# STAND-INS for the real functions in actions.py (no input needed).
def select_client(state):
    print("select_client ran")
    

def create_client(state):
    print("create_client ran")

actions = {
    "select_client": select_client,
    "create_client": create_client,
}

# What load_data.py does: read the menus...
with open("menus.json", "r", encoding="utf-8") as file:
    menus = json.load(file)

ready = all("action" in option for menu in menus.values() for option in menu["options"])
if not ready:
    print("Finish Section 2 (every option needs an action), rerun its cells, then rerun this cell.")
else:
    option = menus["client_management"]["options"][0]
    print("BEFORE binding:", repr(option["action"]), type(option["action"]))

    # ...then bind: the same five lines as load_data.py.
    for menu in menus.values():
        for option in menu["options"]:
            action_name = option["action"]

            if action_name is not None:
                option["action"] = actions[action_name]

    option = menus["client_management"]["options"][0]
    print("AFTER binding: ", option["action"], type(option["action"]))

### The three forms of an action

The key `"action"` stays the same, but its value changes as an option travels from the file to the screen:

| Where | `option["action"]` is | Select Client |
|---|---|---|
| `menus.json` | a **name** (string) | `"select_client"` |
| `actions.py` | a **key**, whose value is the function | `"select_client": select_client` |
| after binding | the **function** itself | ready to call: `option["action"](state)` |

### Check your Question 3C: a missing registration

What if `menus.json` names an action that isn't in the `actions` dictionary? The next cell binds a fresh copy of the menus with `create_client` missing from `actions`.

In [ ]:
incomplete_actions = {"select_client": select_client}    # create_client forgotten

with open("menus.json", "r", encoding="utf-8") as file:
    test_menus = json.load(file)

if not all("action" in option for menu in test_menus.values() for option in menu["options"]):
    print("Finish Section 2 (every option needs an action), rerun its cells, then rerun this cell.")
else:
    try:
        for menu in test_menus.values():
            for option in menu["options"]:
                action_name = option["action"]
                if action_name is not None:
                    option["action"] = incomplete_actions[action_name]
        print("Bound without errors.")
    except KeyError as error:
        print("KeyError:", error)
        print("PEMS stops while loading, before any menu appears.")

When you see `KeyError: 'create_client'` as PEMS starts, the fix is in `actions.py`: the function is missing from the `actions` dictionary, or its key is spelled differently from `menus.json`.

---
## 6. `pems.py`: running the action

Binding put functions into the options. `pems.py` is where they are **called**. Here is the core of the Notebook 1 loop beside the core of the `pems.py` loop:

**Notebook 1 (`navigation_only.py`)**
```python
option = current_menu["options"][int(choice) - 1]

if option["next_menu"] is not None:
    stack.append(option["next_menu"])
```

**`pems.py`**
```python
option = cur_menu["options"][int(choice) - 1]

if option["action"] is not None:              # NEW
    if option["action"](state) is False:      # NEW: call the bound function
        continue                              # NEW: False -> stay on this menu

if option["next_menu"] is not None:
    stack.append(option["next_menu"])
```

The only new idea is those three lines, placed between **selecting** the option and **pushing** the next menu:

- If the option has an action, call it with `state`.
- If the action returns `False`, `continue` skips the push, so the user stays on the current menu.
- Otherwise, navigation works exactly as in Notebook 1.

`option["action"](state)` reads in two steps: `option["action"]` is the function (thanks to binding), and `(state)` calls it.

`pems.py` also has some PEMS polish that isn't new navigation: it clears the selected client whenever you're back on Client Management, shows the client's name on the menus, and says "Enter to exit" on the first menu.

In [ ]:
# QUESTION 4 — Trace pems.py
# A. Select Client has an action AND a next_menu. Why must the action run
#    BEFORE the push?
# B. There are no clients. The user chooses Select Client. What does the
#    action return, and what happens to the stack?
# C. The user chooses Create Client and it succeeds. Does the stack change? Why?
# D. The user chooses Transactions. What runs, and what happens to the stack?

# YOUR ANSWERS:

---
## 7. `state`: what actions work on

`pems.py` calls **every** action the same way: `option["action"](state)`. It can't know that `select_client` needs the client list or that a later `buy_stock` needs prices. So every action receives one dictionary, `state`, holding everything an action might need. This is the same idea as menus as data: one uniform call, with the differences in the data.

Here is how `pems.py` builds it after login:

```python
state = {
    "pm": current_pm,
    "pems": pems,
    "clients": pems[current_pm]["clients"],
    "client_idx": None,
    "trd_day": 1,
    "market_data": market_data,
    "security_info": security_info,
}
```

| Key | Holds | Changed by |
|---|---|---|
| `pm` | the logged-in PM's name | set at login |
| `pems` | all PEMS data; what `save_pems` writes to `pems.json` | indirectly, through `clients` |
| `clients` | this PM's list of client dictionaries | `create_client` appends to it |
| `client_idx` | position of the selected client, or `None` | `select_client` sets it; `pems.py` clears it at Client Management |
| `trd_day` | the current trading day | Advance Trade Day (later) |
| `market_data`, `security_info` | prices and company data | read by later actions |

**Actions report their work by changing `state`.** The return value only says whether to stay on the menu.

### `clients` is not a copy

`"clients": pems[current_pm]["clients"]` does not copy the list. `state["clients"]` and `state["pems"][pm]["clients"]` are two names for the **same** list, just like `alias_101` in readiness exercise C2. Run the next cell to see why that matters for saving.

In [ ]:
pems = {"bob": {"clients": [{"name": "Ada Lovelace"}]}}

state = {
    "pm": "bob",
    "pems": pems,
    "clients": pems["bob"]["clients"],
    "client_idx": None,
}

print("Same list?", state["clients"] is state["pems"]["bob"]["clients"])

state["clients"].append({"name": "Grace Hopper"})     # what create_client does

print("Through state['clients']:", [c["name"] for c in state["clients"]])
print("Through state['pems']:   ", [c["name"] for c in state["pems"]["bob"]["clients"]])

In [ ]:
# QUESTION 5
# A. save_pems(state) writes state["pems"] to pems.json. create_client
#    appends the new client to state["clients"]. Why does pems.json end up
#    containing the new client?
# B. The user selects their second client, then presses Enter from Client
#    Workspace. What is state["client_idx"] now, and which code changed it?

# YOUR ANSWERS:

---
## 8. Homework: finish `create_client`

`create_client` in `actions.py` already prompts for the client's information and rejects bad input with `return False`. **Your job is the two `#TODO`s:** build the client dictionary and its opening transaction.

### The client dictionary

Every client has exactly these keys:

```python
CLIENT_KEYS = ["name", "email", "phone", "transactions"]
```

`"transactions"` starts as an empty list. Until it exists, `create_client` stops with FATAL ERROR: Client record incomplete and nothing is saved.

### The opening transaction

A new client starts with one transaction: the starting cash as a **contribution**. Every transaction in PEMS has exactly these keys, whatever its type:

```python
TRANSACTION_KEYS = ["trans_id", "trd_day", "trans_type", "ticker", "shares",
                    "cost_per_share", "total", "div_amnt",
                    "shares_before", "shares_after"]
```

| Field | Value for the opening contribution |
|---|---|
| `trans_id` | `len(transactions) + 1` |
| `trd_day` | `state['trd_day']` |
| `trans_type` | `"CONTRIBUTION"` |
| `ticker` | `"$$$$"` (cash is treated like a security, priced at 1.0) |
| `shares` | the cash amount |
| `cost_per_share` | `1.0` |
| `total` | the cash amount |
| `div_amnt`, `shares_before`, `shares_after` | `None` |

**Why does every transaction have all ten keys,** even a contribution that uses only seven? Later code (holdings, reports, transaction history) reads transactions of every type. If every transaction has the same keys, that code can read any field without first checking whether it exists; the worst it gets is `None`, never a `KeyError`. Unused fields cost a few `null`s in `pems.json` and buy simpler, safer code everywhere else.

### Test it

Run `pems.py`, log in, and check:

- [ ] Create Client with a blank field, or cash of 0 or text: "Client not created," and you stay on Client Management.
- [ ] Create Client with valid input: "Client Created," and the client appears in `pems.json` with one transaction that has all ten keys.
- [ ] Select Client lists the new client; choosing it opens Client Workspace.
- [ ] Transactions and Reports open their menus; "Not Implemented" does nothing; Enter goes back.
- [ ] Enter from Client Workspace returns to Client Management; Enter there logs out.
- [ ] Log in again: the client is still there.

---
## Takeaway

| Piece | Job |
|---|---|
| `menus.json` | option **names**: where to go (`next_menu`) and what to do (`action`) |
| `actions.py` | the action **functions**, and the `actions` dictionary from name to function |
| `load_data.py` | **binding**: replaces each action name with its function, once, at startup |
| `pems.py` | calls the bound action before navigating; `False` keeps the user on the menu |
| `state` | the one dictionary every action reads and changes |

**The same principle as Notebook 1:** keep the loop general, and put the differences in data — menu names led to menus; now action names lead to functions.